# Multilingual Tokenization Study

This notebook measures token counts for equivalent prompts across 5 major world languages:

- English
- Mandarin Chinese
- Hindi
- Spanish
- French

It compares three usage modes:

1. Classic prompt
2. Vibecoding prompt
3. Agentic prompt

The notebook is designed for live simulation and extension.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from scipy.stats import zscore
import tiktoken

plt.style.use("seaborn-v0_8-darkgrid")

In [ ]:
PROMPTS = {
    "English": {
        "classic": "Explain in 6 bullet points how photosynthesis converts sunlight into chemical energy, then give 2 real-world applications.",
        "vibecoding": "Build a Python CLI called task_timer. Requirements: argparse interface, start/stop commands, JSON persistence, unit tests with pytest, and a concise README. Return code blocks per file.",
        "agent": "You are an autonomous coding agent. Goal: fix failing tests in a web API. Steps: inspect repository, run tests, identify root cause, patch minimally, add regression test, run full suite, summarize risk and rollback plan.",
    },
    "Mandarin Chinese": {
        "classic": "请用6个要点解释光合作用如何把太阳能转化为化学能，然后给出2个现实应用。",
        "vibecoding": "请构建一个名为task_timer的Python命令行工具。要求：使用argparse接口，提供start/stop命令，JSON持久化，用pytest编写单元测试，并提供简洁README。按文件返回代码块。",
        "agent": "你是一个自主编码代理。目标：修复某个Web API中失败的测试。步骤：检查仓库，运行测试，定位根因，最小化补丁，添加回归测试，跑完整测试集，总结风险与回滚方案。",
    },
    "Hindi": {
        "classic": "6 बिंदुओं में समझाइए कि प्रकाश संश्लेषण सूर्य के प्रकाश को रासायनिक ऊर्जा में कैसे बदलता है, फिर 2 वास्तविक अनुप्रयोग दीजिए।",
        "vibecoding": "task_timer नाम का एक Python CLI बनाइए। आवश्यकताएँ: argparse इंटरफ़ेस, start/stop कमांड, JSON persistence, pytest के साथ unit tests, और एक संक्षिप्त README। फ़ाइल-दर-फ़ाइल code blocks लौटाइए।",
        "agent": "आप एक स्वायत्त कोडिंग एजेंट हैं। लक्ष्य: एक web API में failing tests ठीक करना। चरण: repository जाँचें, tests चलाएँ, root cause पहचानें, न्यूनतम patch करें, regression test जोड़ें, पूरा suite चलाएँ, risk और rollback plan बताएं।",
    },
    "Spanish": {
        "classic": "Explica en 6 puntos cómo la fotosíntesis convierte la luz solar en energía química y luego da 2 aplicaciones reales.",
        "vibecoding": "Construye una CLI de Python llamada task_timer. Requisitos: interfaz con argparse, comandos start/stop, persistencia en JSON, pruebas unitarias con pytest y un README conciso. Devuelve bloques de código por archivo.",
        "agent": "Eres un agente autónomo de programación. Objetivo: corregir pruebas fallidas en una API web. Pasos: inspeccionar el repositorio, ejecutar pruebas, identificar la causa raíz, aplicar un parche mínimo, añadir prueba de regresión, ejecutar la suite completa y resumir riesgos y plan de rollback.",
    },
    "French": {
        "classic": "Explique en 6 points comment la photosynthèse transforme la lumière solaire en énergie chimique, puis donne 2 applications concrètes.",
        "vibecoding": "Construis une CLI Python nommée task_timer. Exigences: interface argparse, commandes start/stop, persistance JSON, tests unitaires avec pytest et README concis. Retourne des blocs de code par fichier.",
        "agent": "Tu es un agent de codage autonome. Objectif: corriger des tests en échec dans une API web. Étapes: inspecter le dépôt, lancer les tests, identifier la cause racine, appliquer un correctif minimal, ajouter un test de régression, exécuter toute la suite, résumer les risques et le plan de rollback.",
    },
}

ENCODINGS = ["cl100k_base", "o200k_base"]

In [ ]:
def build_dataframe(prompts: dict, encodings: list[str]) -> pd.DataFrame:
    rows = []
    for language, scenarios in prompts.items():
        for scenario, text in scenarios.items():
            row = {
                "language": language,
                "scenario": scenario,
                "chars": len(text),
                "text": text,
            }
            for enc_name in encodings:
                enc = tiktoken.get_encoding(enc_name)
                row[f"tokens_{enc_name}"] = len(enc.encode(text))
            rows.append(row)
    return pd.DataFrame(rows)


df = build_dataframe(PROMPTS, ENCODINGS)
df

In [ ]:
summary = (
    df.groupby(["scenario", "language"])[["tokens_cl100k_base", "tokens_o200k_base"]]
    .mean()
    .reset_index()
)

pivot_cl = summary.pivot(index="language", columns="scenario", values="tokens_cl100k_base")
pivot_o2 = summary.pivot(index="language", columns="scenario", values="tokens_o200k_base")

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

pivot_cl.plot(kind="bar", ax=axes[0], rot=20, title="cl100k_base")
pivot_o2.plot(kind="bar", ax=axes[1], rot=20, title="o200k_base")

axes[0].set_ylabel("Token count")
axes[1].set_ylabel("")
plt.suptitle("Token count by language and prompt mode", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
def plot_heatmap(matrix: pd.DataFrame, title: str) -> None:
    fig, ax = plt.subplots(figsize=(8, 4.8))
    im = ax.imshow(matrix.values, cmap="viridis", aspect="auto")

    ax.set_xticks(np.arange(matrix.shape[1]))
    ax.set_yticks(np.arange(matrix.shape[0]))
    ax.set_xticklabels(matrix.columns)
    ax.set_yticklabels(matrix.index)
    ax.set_title(title)

    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            ax.text(j, i, int(matrix.values[i, j]), ha="center", va="center", color="white", fontsize=9)

    fig.colorbar(im, ax=ax, label="Tokens")
    plt.tight_layout()
    plt.show()


plot_heatmap(pivot_cl, "Heatmap — cl100k_base")
plot_heatmap(pivot_o2, "Heatmap — o200k_base")

In [ ]:
# Live simulation cell: set `base_prompt` and rerun.
base_prompt = "Design a robust API migration plan with constraints and rollback steps."

language_templates = {
    "English": base_prompt,
    "Mandarin Chinese": "请设计一个稳健的API迁移方案，并包含约束条件和回滚步骤。",
    "Hindi": "कृपया API माइग्रेशन के लिए एक मजबूत योजना बनाइए, जिसमें बाधाएँ और रोलबैक चरण शामिल हों।",
    "Spanish": "Disena un plan robusto de migracion de API con restricciones y pasos de rollback.",
    "French": "Concois un plan robuste de migration d'API avec contraintes et etapes de rollback.",
}

sim_rows = []
for language, text in language_templates.items():
    row = {"language": language, "chars": len(text)}
    for enc_name in ENCODINGS:
        row[f"tokens_{enc_name}"] = len(tiktoken.get_encoding(enc_name).encode(text))
    sim_rows.append(row)

sim_df = pd.DataFrame(sim_rows).set_index("language")
sim_df["z_cl100k"] = zscore(sim_df["tokens_cl100k_base"])
sim_df["z_o200k"] = zscore(sim_df["tokens_o200k_base"])

display(sim_df)

ax = sim_df[["tokens_cl100k_base", "tokens_o200k_base"]].plot(kind="bar", figsize=(9, 4), rot=20)
ax.set_ylabel("Tokens")
ax.set_title("Live simulation — token count by language")
plt.tight_layout()
plt.show()